# EVA Foundry Library - Quick Start Guide

**Version:** 0.1.0-beta  
**Last Updated:** February 18, 2026 - 7:45 AM ET  
**Duration:** 20-30 minutes

This notebook provides a hands-on introduction to the EVA Foundry Library, covering:
1. Environment setup and authentication
2. Creating your first agent
3. Using tools from the foundry library
4. Integrating MCP servers
5. Loading and using Prompty templates

## Prerequisites

Before running this notebook:
- Python 3.10+ installed
- Azure subscription with Azure OpenAI access
- VS Code with Jupyter extension (or Jupyter Lab)
- Completed setup: `pip install -r requirements.txt`

## 1. Setup and Configuration

In [ ]:
# Import core libraries
import os
import sys
import asyncio
from pathlib import Path

# Add foundry library to path
foundry_path = Path.cwd().parent
sys.path.insert(0, str(foundry_path))

print(f"✅ Foundry library path: {foundry_path}")
print(f"✅ Python version: {sys.version}")

In [ ]:
# Load environment variables
from dotenv import load_dotenv

env_file = foundry_path / ".env"
if env_file.exists():
    load_dotenv(env_file)
    print("✅ Environment variables loaded")
else:
    print("⚠️  No .env file found. Set environment variables manually.")

In [ ]:
# Verify Azure configuration
required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_SEARCH_ENDPOINT",
]

missing_vars = [var for var in required_vars if not os.getenv(var)]

if missing_vars:
    print(f"❌ Missing environment variables: {', '.join(missing_vars)}")
    print("\nPlease set them in .env file or environment.")
else:
    print("✅ All required environment variables set")
    print(f"   Azure OpenAI: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
    print(f"   Azure Search: {os.getenv('AZURE_SEARCH_ENDPOINT')}")

## 2. Azure Authentication

In [ ]:
# Setup Azure authentication
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# DefaultAzureCredential automatically handles:
# - Environment variables
# - Managed Identity (production)
# - VS Code Azure Account
# - Azure CLI credentials

credential = DefaultAzureCredential()

# Test authentication
try:
    token = credential.get_token("https://cognitiveservices.azure.com/.default")
    print("✅ Azure authentication successful")
    print(f"   Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("\nTroubleshooting:")
    print("1. Run 'az login' in terminal")
    print("2. Or sign in via VS Code Azure extension")
    print("3. Or set AZURE_CLIENT_ID, AZURE_CLIENT_SECRET, AZURE_TENANT_ID")

## 3. Create Your First Agent

In [ ]:
# Import Agent Framework
from agent_framework import Agent

# Create a simple agent
agent = Agent(
    name="SimpleAssistant",
    instructions="""You are a helpful assistant that provides clear, concise answers.
    Be friendly and professional.""",
    model="gpt-4o",
    temperature=0.7
)

print("✅ Agent created:", agent.name)

In [ ]:
# Run the agent (simple query)
async def test_agent():
    response = await agent.run(
        "What is Employment Insurance in Canada?",
        thread_id="quickstart-demo"
    )
    return response

# Execute
response = await test_agent()

print("\n📝 Agent Response:")
print(response.content)
print(f"\n🔗 Thread ID: {response.thread_id}")

In [ ]:
# Continue the conversation (agent remembers context)
async def continue_conversation():
    response = await agent.run(
        "How do I apply for it?",  # Refers to EI from previous message
        thread_id="quickstart-demo"  # Same thread ID
    )
    return response

response2 = await continue_conversation()

print("\n📝 Agent Response (Turn 2):")
print(response2.content)

## 4. Using Foundry Tools

In [ ]:
# Import foundry tools
from tools.search import EVASearchClient
from tools.auth import get_azure_credential

# Initialize search client
search_client = EVASearchClient(
    endpoint=os.getenv("AZURE_SEARCH_ENDPOINT"),
    index_name="ei-policies",  # Your index name
    credential=credential
)

print("✅ Search client initialized")

In [ ]:
# Test search functionality
async def test_search():
    results = await search_client.hybrid_search(
        query="EI eligibility requirements",
        top_k=5
    )
    return results

search_results = await test_search()

print(f"\n🔍 Found {len(search_results)} results:")
for i, result in enumerate(search_results, 1):
    print(f"\n{i}. {result.get('title', 'Untitled')}")
    print(f"   Score: {result.get('@search.score', 0):.2f}")
    print(f"   Section: {result.get('section', 'N/A')}")

## 5. Agent with Tools

In [ ]:
# Define tool function for agent
async def search_ei_policies(query: str, top_k: int = 5) -> str:
    """
    Search Employment Insurance policy documents.
    
    Args:
        query: Search query text
        top_k: Number of results to return (default: 5)
    
    Returns:
        JSON string with search results
    """
    import json
    
    results = await search_client.hybrid_search(query=query, top_k=top_k)
    
    # Format for agent
    formatted = []
    for r in results:
        formatted.append({
            "title": r.get("title", "Untitled"),
            "content": r.get("content", "")[:200],  # First 200 chars
            "section": r.get("section", "N/A"),
            "score": round(r.get("@search.score", 0), 2)
        })
    
    return json.dumps(formatted, indent=2)

# Create agent with tool
policy_agent = Agent(
    name="PolicyExpert",
    instructions="""You are an expert on Employment Insurance policies.
    Use the search_ei_policies tool to find accurate information.
    Always cite your sources.""",
    model="gpt-4o",
    temperature=0.3,  # Lower temperature for factual responses
    tools=[search_ei_policies]
)

print("✅ Policy agent with search tool created")

In [ ]:
# Test agent with tool
async def test_policy_agent():
    response = await policy_agent.run(
        "What are the minimum hours required for EI eligibility?",
        thread_id="policy-demo"
    )
    return response

response = await test_policy_agent()

print("\n📝 Policy Agent Response:")
print(response.content)
print("\n🔧 Tools Used:", response.tool_calls if hasattr(response, 'tool_calls') else "None")

## 6. Loading Prompty Templates

In [ ]:
# Load a Prompty template
from prompty import load_prompty

prompt_file = foundry_path / "prompts" / "policy-analysis.prompty"

if prompt_file.exists():
    policy_prompt = load_prompty(str(prompt_file))
    print("✅ Prompty template loaded")
    print(f"   Name: {policy_prompt.name}")
    print(f"   Model: {policy_prompt.model}")
    print(f"   Temperature: {policy_prompt.temperature}")
else:
    print(f"❌ Prompty file not found: {prompt_file}")

In [ ]:
# Use prompty template with agent
prompty_agent = Agent(
    name="PolicyAnalyst",
    instructions=policy_prompt.system_prompt,  # From prompty file
    model=policy_prompt.model,
    temperature=policy_prompt.temperature
)

# Render user prompt with variables
user_prompt = policy_prompt.render_user(
    question="Who qualifies for EI regular benefits?",
    context="",
    search_results=[]
)

print("✅ Agent created from Prompty template")
print(f"\n📋 Rendered User Prompt (first 200 chars):\n{user_prompt[:200]}...")

## 7. Quick Summary

### What You've Learned

1. ✅ **Setup:** Environment configuration and Azure authentication
2. ✅ **Basic Agent:** Created and ran a simple conversational agent
3. ✅ **Conversation History:** Maintained context across multiple turns
4. ✅ **Foundry Tools:** Used EVASearchClient for policy search
5. ✅ **Agent Tools:** Created agents that call functions
6. ✅ **Prompty:** Loaded and used Prompty templates

### Next Steps

- **Notebook 02:** Multi-agent workflows and orchestration
- **Notebook 03:** RAG pipelines with Azure AI Search
- **Notebook 04:** Evaluation and testing

### Key Concepts

- **Agent:** Autonomous entity with instructions and model
- **Thread:** Conversation session with message history
- **Tools:** Functions agents can call
- **Prompty:** Version-controlled prompt templates

## 8. Cleanup (Optional)

In [ ]:
# Close connections
if 'search_client' in locals():
    await search_client.close()
    print("✅ Search client closed")

print("\n🎉 Quick start complete!")